# Load and Extract Raw ECG Data from Apple Health

This notebook will:
1. Load the export.xml file
2. Extract all data related to hear rate
3. Calculate and analyze

## 1. Import necessary libraries

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import timedelta
import os
import re
from tqdm import tqdm  # Progress bar

print("✅ Import successful!")

✅ Import successful!


## 2. File path

In [2]:
# Path to export.xml file
xml_file_path = r"C:\Project\Apple Health Data\data\apple_health_export\export.xml"
temp_file = r"C:\Project\Apple Health Data\data\apple_health_export\export_cleaned.xml"

# Check if file exists
if os.path.exists(xml_file_path):
    file_size = os.path.getsize(xml_file_path) / (1024 * 1024)
    print(f"✅ File found!")
    print(f"📊 File size: {file_size:.2f} MB")
else:
    print(f"❌ File not found at: {xml_file_path}")
    print("⚠️  Please check the path again!")

✅ File found!
📊 File size: 1056.52 MB


## 3. Load XML file and parse data

In [3]:
print("⏳ Step 1: Removing DTD and fixing ALL duplicate attributes...")
print("="*60)

def remove_duplicate_attrs(line):
    """Remove duplicate attributes from any XML tag"""
    def fix_tag(match):
        tag_content = match.group(0)
        # Tìm tất cả attr="value" pairs
        attrs = re.findall(r'(\w+)="([^"]*)"', tag_content)
        seen = {}
        for attr, value in attrs:
            if attr not in seen:
                seen[attr] = value
        # Rebuild tag
        tag_name = re.match(r'<(\w+)', tag_content).group(1)
        is_self_closing = tag_content.rstrip().endswith('/>')
        new_attrs = ' '.join(f'{k}="{v}"' for k, v in seen.items())
        if is_self_closing:
            return f'<{tag_name} {new_attrs}/>'
        else:
            return f'<{tag_name} {new_attrs}>'
    
    # Match any opening tag with attributes
    return re.sub(r'<\w+\s+[^>]+/?>', fix_tag, line)

with open(xml_file_path, 'r', encoding='utf-8') as infile:
    with open(temp_file, 'w', encoding='utf-8') as outfile:
        in_doctype = False
        bracket_count = 0
        line_num = 0
        
        for line in infile:
            line_num += 1
            
            # Bỏ DOCTYPE
            if '<!DOCTYPE' in line:
                in_doctype = True
            if in_doctype:
                bracket_count += line.count('[')
                bracket_count -= line.count(']')
                if ']>' in line or (bracket_count <= 0 and '>' in line):
                    in_doctype = False
                continue
            
            # Fix duplicate attributes trong mọi tag
            if '<' in line and '="' in line:
                line = remove_duplicate_attrs(line)
            
            outfile.write(line)
            
            if line_num % 1000000 == 0:
                print(f"  Cleaned {line_num:,} lines...")

print("✅ File cleaned.")
print()
print("⏳ Step 2: Parsing cleaned file...")

record_count = 0
for event, elem in ET.iterparse(temp_file, events=('end',)):
    if elem.tag == 'Record':
        record_count += 1
        elem.clear()
    
    if record_count % 500000 == 0 and record_count > 0:
        print(f"  Processed {record_count:,} records...")

print("="*60)
print(f"✅ Done! Total records: {record_count:,}")

⏳ Step 1: Removing DTD and fixing ALL duplicate attributes...
  Cleaned 1,000,000 lines...
  Cleaned 2,000,000 lines...
  Cleaned 3,000,000 lines...
✅ File cleaned.

⏳ Step 2: Parsing cleaned file...
  Processed 500,000 records...
  Processed 1,000,000 records...
  Processed 1,500,000 records...
  Processed 2,000,000 records...
  Processed 2,500,000 records...
✅ Done! Total records: 2,713,541


## 4. Extract raw heart rate data

In [4]:
print("⏳ Searching for Heart Rate data types...")
print("="*60)

heart_rate_types = {}
record_count = 0

for event, elem in ET.iterparse(temp_file, events=('end',)):
    if elem.tag == 'Record':
        record_count += 1
        record_type = elem.get('type', '')
        
        if 'heart' in record_type.lower() or 'cardio' in record_type.lower():
            if record_type not in heart_rate_types:
                heart_rate_types[record_type] = {
                    'count': 0,
                    'sample': dict(elem.attrib)
                }
            heart_rate_types[record_type]['count'] += 1
        
        elem.clear()
        
        if record_count % 500000 == 0:
            print(f"  Scanned {record_count:,} records...")

print("="*60)
print(f"✅ Scanned {record_count:,} total records\n")

print(f"📊 Found {len(heart_rate_types)} Heart Rate data types:\n")
for record_type, info in sorted(heart_rate_types.items(), key=lambda x: x[1]['count'], reverse=True):
    print(f"✅ {record_type}")
    print(f"   Count: {info['count']:,} records")
    sample_preview = dict(list(info['sample'].items())[:3])
    print(f"   Example: {sample_preview}")
    print()

⏳ Searching for Heart Rate data types...
  Scanned 500,000 records...
  Scanned 1,000,000 records...
  Scanned 1,500,000 records...
  Scanned 2,000,000 records...
  Scanned 2,500,000 records...
✅ Scanned 2,713,541 total records

📊 Found 5 Heart Rate data types:

✅ HKQuantityTypeIdentifierHeartRate
   Count: 356,862 records
   Example: {'type': 'HKQuantityTypeIdentifierHeartRate', 'sourceName': 'Matthew’s Apple\xa0Watch', 'sourceVersion': '3.2'}

✅ HKQuantityTypeIdentifierHeartRateVariabilitySDNN
   Count: 4,885 records
   Example: {'type': 'HKQuantityTypeIdentifierHeartRateVariabilitySDNN', 'sourceName': 'Matthew’s Apple\xa0Watch', 'sourceVersion': '4.1'}

✅ HKQuantityTypeIdentifierRestingHeartRate
   Count: 1,759 records
   Example: {'type': 'HKQuantityTypeIdentifierRestingHeartRate', 'sourceName': 'Matthew’s Apple\xa0Watch', 'sourceVersion': '4.1'}

✅ HKQuantityTypeIdentifierWalkingHeartRateAverage
   Count: 1,661 records
   Example: {'type': 'HKQuantityTypeIdentifierWalkingHeartRate

In [5]:
import os
import glob
import xml.etree.ElementTree as ET
# Find all GPX files
gpx_folder = r"C:\Project\Apple Health Data\data\apple_health_export\workout-routes"
gpx_files = glob.glob(os.path.join(gpx_folder, "*.gpx"))
print(f"📊 Found {len(gpx_files)} GPX workout route files")
# Read 1 sample file
if len(gpx_files) > 0:
    sample_file = gpx_files[0]
    print(f"\n📝 Analyzing file: {os.path.basename(sample_file)}")
    
    tree = ET.parse(sample_file)
    root = tree.getroot()
    
    print(f"   Root tag: {root.tag}")
    
    # Find all tags (without namespace)
    all_tags = set()
    for elem in root.iter():
        tag_name = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        all_tags.add(tag_name)
    
    print(f"\n   📋 Tags in the file:")
    for tag in sorted(all_tags):
        count = len([e for e in root.iter() if e.tag.endswith(tag)])
        print(f"      - {tag}: {count}")
    
    # Find track points (remove namespace)
    trkpts = [elem for elem in root.iter() if elem.tag.endswith('trkpt')]
    print(f"\n   📍 Track points: {len(trkpts)}")
    
    if len(trkpts) > 0:
        first_pt = trkpts[0]
        print(f"\n   First track point:")
        print(f"      Attributes: {first_pt.attrib}")
        print(f"      Children:")
        for child in first_pt:
            tag_name = child.tag.split('}')[-1] if '}' in child.tag else child.tag
            print(f"         {tag_name}: {child.text}")

📊 Found 177 GPX workout route files

📝 Analyzing file: route_2018-01-07_4.49pm.gpx
   Root tag: {http://www.topografix.com/GPX/1/1}gpx

   📋 Tags in the file:
      - course: 8916
      - ele: 8916
      - extensions: 8916
      - gpx: 1
      - hAcc: 8916
      - metadata: 1
      - name: 1
      - speed: 8916
      - time: 8917
      - trk: 1
      - trkpt: 8916
      - trkseg: 1
      - vAcc: 8916

   📍 Track points: 8916

   First track point:
      Attributes: {'lon': '138.597218', 'lat': '-35.004800'}
      Children:
         ele: 138.675110
         time: 2018-01-07T03:39:50Z
         extensions: None


In [6]:
print("💓 Loading ALL Heart Rate data and Workouts...")
print("="*60)

hr_data_full = []
workout_list = []
record_count = 0

for event, elem in ET.iterparse(temp_file, events=('end',)):
    if elem.tag == 'Record':
        record_count += 1
        if elem.get('type') == 'HKQuantityTypeIdentifierHeartRate':
            hr_data_full.append({
                'datetime': elem.get('startDate'),
                'value': float(elem.get('value', 0))
            })
        elem.clear()
        
        if record_count % 500000 == 0:
            print(f"  Scanned {record_count:,} records...")
    
    elif elem.tag == 'Workout':
        distance_val = elem.get('totalDistance')
        duration_val = elem.get('duration')
        workout_list.append({
            'type': elem.get('workoutActivityType'),
            'start': elem.get('startDate'),
            'end': elem.get('endDate'),
            'duration': float(duration_val) if duration_val else 0.0,
            'distance': float(distance_val) if distance_val else 0.0,
        })
        elem.clear()

print("="*60)
print(f"✅ Scanned {record_count:,} total records")

# Process Heart Rate data
df_hr_full = pd.DataFrame(hr_data_full)

if len(df_hr_full) > 0:
    df_hr_full['datetime'] = pd.to_datetime(df_hr_full['datetime'])
    df_hr_full = df_hr_full.sort_values('datetime')
    print(f"💓 Loaded {len(df_hr_full):,} HR records")
    print(f"   Time range: {df_hr_full['datetime'].min()} to {df_hr_full['datetime'].max()}")
    print(f"   HR range: {df_hr_full['value'].min():.0f} - {df_hr_full['value'].max():.0f} bpm\n")
else:
    print(f"❌ No heart rate data found!")

# Process Workouts
print(f"🏃 Found {len(workout_list):,} workout records")
df_workouts_full = pd.DataFrame(workout_list)

if len(df_workouts_full) > 0:
    df_workouts_full['start_time'] = pd.to_datetime(df_workouts_full['start'])
    df_workouts_full['end_time'] = pd.to_datetime(df_workouts_full['end'])
    
    if len(df_hr_full) > 0:
        hr_start = df_hr_full['datetime'].min()
        hr_end = df_hr_full['datetime'].max()
        df_workouts_with_hr = df_workouts_full[
            (df_workouts_full['start_time'] >= hr_start) & 
            (df_workouts_full['end_time'] <= hr_end)
        ]
        print(f"✅ Workouts with HR data: {len(df_workouts_with_hr)}")
        print(f"   From {hr_start} to {hr_end}\n")
        
        if len(df_workouts_with_hr) > 0:
            print("🏃 10 most recent workouts:")
            recent_workouts = df_workouts_with_hr.sort_values('start_time', ascending=False).head(10)
            for idx, (_, row) in enumerate(recent_workouts.iterrows(), 1):
                print(f"   {idx}. {row['type']:<35} | {row['start_time'].strftime('%Y-%m-%d %H:%M')} | {row['duration']:.1f} min")
            
            print("\n📌 Longest workout:")
            longest = df_workouts_with_hr.loc[df_workouts_with_hr['duration'].idxmax()]
            print(f"   Type: {longest['type']}")
            print(f"   Duration: {longest['duration']:.1f} min")
            print(f"   Start: {longest['start_time']}")
        else:
            longest = None
    else:
        df_workouts_with_hr = pd.DataFrame()
        longest = None
else:
    print("❌ No workouts found!")
    df_workouts_with_hr = pd.DataFrame()
    longest = None

💓 Loading ALL Heart Rate data and Workouts...
  Scanned 500,000 records...
  Scanned 1,000,000 records...
  Scanned 1,500,000 records...
  Scanned 2,000,000 records...
  Scanned 2,500,000 records...
✅ Scanned 2,713,541 total records
💓 Loaded 356,862 HR records
   Time range: 2017-11-27 09:47:34+09:30 to 2022-09-23 10:59:44+09:30
   HR range: 41 - 199 bpm

🏃 Found 289 workout records
✅ Workouts with HR data: 289
   From 2017-11-27 09:47:34+09:30 to 2022-09-23 10:59:44+09:30

🏃 10 most recent workouts:
   1. HKWorkoutActivityTypeRowing         | 2022-09-22 10:11 | 33.1 min
   2. HKWorkoutActivityTypeRowing         | 2022-09-19 14:17 | 175.3 min
   3. HKWorkoutActivityTypeWalking        | 2022-09-09 14:42 | 10.9 min
   4. HKWorkoutActivityTypeRowing         | 2022-08-30 20:05 | 51.6 min
   5. HKWorkoutActivityTypeWalking        | 2022-08-27 16:44 | 3.2 min
   6. HKWorkoutActivityTypeWalking        | 2022-08-25 20:07 | 6.0 min
   7. HKWorkoutActivityTypeRunning        | 2022-08-16 22:54 | 